# Construindo uma Rede Neural

### Cenário
Uma empresa de e-commerce deseja estimar a **probabilidade de atraso de uma entrega**.

Cada pedido será representado por três características já normalizadas:

- `distancia`
- `quantidade_itens`
- `frete`

O alvo é:

\
y =
\begin{cases}
0 & \text{entrega no prazo}\\
1 & \text{entrega atrasada}
\end{cases}


### Objetivo

Ao longo desta atividade, você irá construir uma pequena rede neural para classificação binária só utilizando tensorflow.

Ao final, deverá compreender o fluxo:

> **Regra da aula:** não utilize `model.fit()`. Hoje queremos entender como a rede **produz uma predição**. O treinamento será estudado na próxima aula.

## Preparação

Execute a célula abaixo para importar as bibliotecas necessárias.

In [1]:
import tensorflow as tf
import numpy as np

print("TensorFlow:", tf.__version__)

TensorFlow: 2.20.0


# Tarefa 1: Representando os dados

| Pedido | Distância | Quantidade de itens | Frete | Atrasou? |
|---|---:|---:|---:|---:|
| 1 | 0.20 | 0.10 | 0.30 | 0 |
| 2 | 0.80 | 0.70 | 0.90 | 1 |
| 3 | 0.30 | 0.20 | 0.40 | 0 |
| 4 | 0.90 | 0.80 | 0.70 | 1 |
| 5 | 0.15 | 0.30 | 0.20 | 0 |
| 6 | 0.75 | 0.60 | 0.85 | 1 |

`X` contém as três features e `y` contém o alvo.

**Shape de `X`:** `(6, 3)`

**Shape de `y`:** `(6, 1)`

In [ ]:
X = tf.constant([
    [0.20, 0.10, 0.30],
    [0.80, 0.70, 0.90],
    [0.30, 0.20, 0.40],
    [0.90, 0.80, 0.70],
    [0.15, 0.30, 0.20],
    [0.75, 0.60, 0.85]
], dtype=tf.float32)

y = tf.constant([[0.0], [1.0], [0.0], [1.0], [0.0], [1.0]], dtype=tf.float32)

print(X.shape)
print(y.shape)

### Verificação conceitual

Cada **linha** de `X` representa um pedido.

Cada **coluna** de `X` representa uma característica do pedido.

O valor `y = 1` significa que a entrega atrasou.

# Tarefa 2: O que acontece dentro de um neurônio?

Para o primeiro pedido:

$$
z = (0.20 \times 0.5) + (0.10 \times -0.3) + (0.30 \times 0.8) + 0.1
$$

$$
\boxed{z = 0.31}
$$

O valor de $z$ não representa diretamente uma probabilidade; ele é o valor bruto antes da função de ativação.

In [ ]:
w = tf.constant([0.5, -0.3, 0.8], dtype=tf.float32)
b = tf.constant(0.1, dtype=tf.float32)
x = tf.constant([0.20, 0.10, 0.30], dtype=tf.float32)
z = tf.reduce_sum(x * w) + b
print("z =", z.numpy())

### Pergunta

O valor de $z$ representa diretamente a probabilidade de atraso?

**( ) Sim**

**(X) Não**

Justifique:

$z$ é o valor bruto da combinação linear. É necessário aplicar uma função sigmoid para obter uma probabilidade entre 0 e 1.

# Tarefa 3: Função de ativação

A ReLU é definida por:

$$
\boxed{\operatorname{ReLU}(z)=\max(0,z)}
$$

| $z$ | $\operatorname{ReLU}(z)$ |
|---:|---:|
| -3 | 0 |
| -0.5 | 0 |
| 0 | 0 |
| 2 | 2 |
| 5 | 5 |

Uma rede neural precisa de funções de ativação não lineares para aprender relações complexas.

In [ ]:
valores = tf.constant([-3.0, -0.5, 0.0, 2.0, 5.0])
resultado = tf.nn.relu(valores)
print(resultado.numpy())

### Pense antes de continuar

Por que uma rede neural precisa de funções de ativação **não lineares**?

Porque sem não linearidade várias camadas equivaleriam a uma única transformação linear e a rede não aprenderia relações complexas.

# Tarefa 4: Projetando a arquitetura

A arquitetura da rede é:

$$
\boxed{3 \rightarrow 4 \rightarrow 1}
$$

O fluxo é:

$$
X
\rightarrow
\boxed{\text{Dense}(4)}
\rightarrow
\boxed{\text{ReLU}}
\rightarrow
\boxed{\text{Dense}(1)}
\rightarrow
\boxed{\text{Sigmoid}}
\rightarrow
\hat{y}
$$

### Agora construa a rede

Use:

- `tf.keras.Sequential`
- `tf.keras.layers.Input`
- `tf.keras.layers.Dense`

> **Dica:** a primeira camada `Dense` deve ter 4 neurônios e a segunda deve ter 1.


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(3,)),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

# Tarefa 5: Quantos parâmetros existem?

Nossa rede possui a arquitetura:

$$
\boxed{3 \rightarrow 4 \rightarrow 1}
$$

## 1. Camada escondida

$ n_{in} = 3 $

$ n_{out} = 4 $

Pesos: $3 \times 4 = 12$

Biases: $4$

Total: $12 + 4 = \boxed{16}$

## 2. Camada de saída

$ n_{in} = 4 $

$ n_{out} = 1 $

Pesos: $4 \times 1 = 4$

Biases: $1$

Total: $4 + 1 = \boxed{5}$

## 3. Total da rede

$$
\boxed{\text{Total de parâmetros} = 16 + 5 = 21}
$$

O resultado esperado em `model.summary()` é **21 parâmetros**.

In [ ]:
model.summary()

### Conferência

Seu cálculo coincidiu com o TensorFlow?

**(X) Sim**

**( ) Não**

O cálculo é $16 + 5 = 21$ parâmetros.

# Tarefa 6: Forward Pass

O forward pass leva os dados da entrada até a saída:

$$
X
\rightarrow
\text{Dense}(4)
\rightarrow
\text{ReLU}
\rightarrow
\text{Dense}(1)
\rightarrow
\text{Sigmoid}
\rightarrow
\hat{y}
$$

`y_pred` representa a probabilidade estimada de atraso para cada pedido.

In [ ]:
y_pred = model(X)
print(y_pred.numpy())

# Tarefa 7: Probabilidade não é classe

Após o forward pass, a saída da rede é uma probabilidade entre 0 e 1.

## A. O que significa $\hat{y}=0.78$?

A rede estima 78% de probabilidade de o pedido atrasar.

## B. Transformando probabilidade em classe

Com threshold $0.5$, a classe prevista é:

$$
\boxed{\text{classe}=1}
$$

## C. E se alterarmos o threshold?

Com threshold $0.85$ e probabilidade $0.78$:

$$
\boxed{\text{classe}=0}
$$

## D. A rede mudou?

- [ ] Sim
- [X] Não

Os pesos e biases não foram modificados; apenas a regra de decisão mudou.

## Complete o fluxo

A rede neural termina na produção da probabilidade. O threshold inicia a regra de decisão que transforma a probabilidade em classe.

## Aplicando um threshold

Agora transforme as probabilidades da sua rede em classes usando threshold `0.5`.

In [ ]:
threshold = 0.5
y_class = tf.cast(y_pred >= threshold, tf.int32)

print("Probabilidades:")
print(y_pred.numpy())

print("\nClasses:")
print(y_class.numpy())

# Tarefa 8: Comparando previsão e realidade

Agora exiba, para cada pedido:

- valor real;
- probabilidade prevista;
- classe prevista.

Complete o código.


In [ ]:
for i in range(len(y)):
    real = int(y[i].numpy()[0])
    probabilidade = float(y_pred[i].numpy()[0])
    classe = int(y_class[i].numpy()[0])

    print(
        f"Pedido {i+1}: "
        f"real={real} | "
        f"probabilidade={probabilidade:.3f} | "
        f"classe={classe}"
    )

# Tarefa 9: Como medir o erro?

Para $y=1$:

A. Melhor previsão: **Modelo A**, com probabilidade 0.95.

B. Pior previsão: **Modelo C**, com probabilidade 0.05.

C. Maior penalização: **Modelo C**.

D. Quanto mais próxima de 1 estiver a probabilidade prevista, menor será o erro. Uma previsão baixa e confiante para um caso positivo deve receber uma penalização maior.

A função que mede esse erro é a **Loss**, especialmente a Binary Cross-Entropy em classificação binária.

## Binary Cross-Entropy

Na classificação binária, a Binary Cross-Entropy mede o erro das probabilidades previstas.

$$
L = -\left[y\log(\hat{y}) + (1-y)\log(1-\hat{y})\right]
$$

Para $y=1$:

$$
L=-\log(\hat{y})
$$

Quanto mais próxima de 1 for a probabilidade correta, menor será a loss.

In [ ]:
loss_fn = tf.keras.losses.BinaryCrossentropy()
y_true = tf.constant([[1.0]])
predicoes = [
    tf.constant([[0.95]]),
    tf.constant([[0.60]]),
    tf.constant([[0.05]])
]

for predicao in predicoes:
    loss = loss_fn(y_true, predicao)
    print(
        "Predição:",
        float(predicao.numpy()[0, 0]),
        "| Loss:",
        float(loss.numpy())
    )

### Interpretação

Quanto **melhor** a previsão, geralmente **menor** será a loss.

Quanto mais **errada e confiante** a previsão, geralmente **maior** será a loss.

# Tarefa 10: Loss da nossa rede

Agora utilize:

- os valores verdadeiros `y`;
- as probabilidades `y_pred`;

para calcular a Binary Cross-Entropy da nossa rede.

> **Importante:** utilize `y_pred`, e não as classes obtidas após o threshold.


In [ ]:
loss_fn = tf.keras.losses.BinaryCrossentropy()
loss = loss_fn(y, y_pred)
print("Loss:", float(loss.numpy()))

# Desafio final: Como a rede aprende?

1. A rede calcula gradientes para descobrir se cada peso deve aumentar ou diminuir.

2. O gradiente indica quanto cada peso deve mudar.

3. A comparação da loss atual com as losses anteriores mostra se as mudanças estão melhorando o modelo.

$$
\boxed{
\text{Loss}
\rightarrow
\text{Gradientes}
\rightarrow
\text{Backpropagation}
\rightarrow
\text{Gradient Descent}
\rightarrow
\text{Atualização dos parâmetros}
}
$$

---

# Checklist de aprendizagem

- [X] O que é um neurônio artificial.
- [X] O papel de pesos e bias.
- [X] O que representa $z=Wx+b$.
- [X] Por que utilizamos funções de ativação.
- [X] Quando utilizar ReLU.
- [X] Por que utilizamos sigmoid na saída deste problema.
- [X] O que significa `Dense(4)`.
- [X] Como calcular o número de parâmetros de uma camada Dense.
- [X] O que é forward pass.
- [X] Diferença entre probabilidade e classe.
- [X] O papel do threshold.
- [X] O que é uma função de loss.
- [X] Por que a rede ainda não está aprendendo neste notebook.

## Próxima aula

**Como uma rede neural aprende?**